# linalg-solve-batched composite — cx3: solve t in P=O+tD for ray-plane intersection, then evaluate P

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `linalg-solve-batched`, `ray-parametric-form`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "linalg-solve-batched"
DD_ATOM_IDS = ["linalg-solve-batched", "ray-parametric-form"]
DD_SUBTOPICS = ["PyTorch: Batched linalg.solve", "Geometry: Ray parametric form"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Ray-plane intersection takes both atoms in one turn. The plane is `n · (P - Q) = 0` (normal `n`, point on plane `Q`); the ray is `P(t) = O + t*D` (the `ray-parametric-form` atom). Substituting:

  `n · (O + t*D - Q) = 0`  ⇒  `t = (n · (Q - O)) / (n · D)`

That's a scalar 1x1 linear solve per ray. For a batch of `NR` rays, that's a `(NR, 1, 1)` coefficient matrix and a `(NR, 1)` RHS — the canonical `linalg-solve-batched` shape contract. After solving, we plug `t` back into the parametric form to recover the 3D intersection point `P = O + t*D` — that's the second use of `ray-parametric-form`.

The whole pipeline is two atom-calls deep: solve for `t`, then evaluate `P(t)`. No loops.

### Composite Exercise — solve t in P=O+tD for ray-plane intersection, then evaluate P

**Atoms exercised together**: `linalg-solve-batched`, `ray-parametric-form`

Implement `cx3_ray_plane_intersect(rays, plane_normal, plane_point)` that computes the intersection points of `NR` rays with a single plane.

- `rays` has shape `(NR, 2, 3)`: `rays[r, 0]` is origin `O_r`, `rays[r, 1]` is direction `D_r`.
- `plane_normal` has shape `(3,)`: the plane normal `n`.
- `plane_point` has shape `(3,)`: a point `Q` on the plane.

Return `(t_vals, points)`:
- `t_vals: (NR,)` — the ray parameter at intersection.
- `points: (NR, 3)` — the 3D intersection points `O + t*D`.

**Algorithm.**
1. Build a `(NR, 1, 1)` coefficient matrix `A[r] = [[n · D_r]]` and a `(NR, 1)` RHS `b[r] = [n · (Q - O_r)]`.
2. `t.linalg.solve(A, b)` → `(NR, 1)`. Squeeze to `(NR,)`.
3. Evaluate `P = O + t * D` per ray (the `ray-parametric-form` atom) — broadcast `t: (NR, 1)` against `D: (NR, 3)`.

Assume no rays are parallel to the plane (no near-zero `n · D` — the singular case is a separate drill).

In [ ]:
def cx3_ray_plane_intersect(rays, plane_normal, plane_point):
    O = rays[:, 0]              # (NR, 3)
    D = rays[:, 1]              # (NR, 3)
    NR = O.shape[0]
    # Build (NR, 1, 1) coefficient: n . D per ray.
    nd = (D * plane_normal).sum(dim=-1)             # (NR,)
    A = nd.reshape(NR, 1, 1)
    # Build (NR, 1) RHS: n . (Q - O) per ray.
    rhs = ((plane_point - O) * plane_normal).sum(dim=-1).reshape(NR, 1)
    # linalg-solve-batched over (NR, 1, 1) systems.
    t_solved = t.linalg.solve(A, rhs).squeeze(-1)   # (NR,)
    # ray-parametric-form: P = O + t * D, broadcast t:(NR,1) against D:(NR,3).
    points = O + t_solved.unsqueeze(-1) * D
    return t_solved, points


<details><summary>Show solution — cx3</summary>

```python
def cx3_ray_plane_intersect(rays, plane_normal, plane_point):
    O = rays[:, 0]              # (NR, 3)
    D = rays[:, 1]              # (NR, 3)
    NR = O.shape[0]
    # Build (NR, 1, 1) coefficient: n . D per ray.
    nd = (D * plane_normal).sum(dim=-1)             # (NR,)
    A = nd.reshape(NR, 1, 1)
    # Build (NR, 1) RHS: n . (Q - O) per ray.
    rhs = ((plane_point - O) * plane_normal).sum(dim=-1).reshape(NR, 1)
    # linalg-solve-batched over (NR, 1, 1) systems.
    t_solved = t.linalg.solve(A, rhs).squeeze(-1)   # (NR,)
    # ray-parametric-form: P = O + t * D, broadcast t:(NR,1) against D:(NR,3).
    points = O + t_solved.unsqueeze(-1) * D
    return t_solved, points
```

The 1x1 batched solve looks like overkill for a scalar division, and it IS — you could just write `t = rhs / nd`. But the point of this drill is the SHAPE DISCIPLINE: batched solve's `(K, n, n)` / `(K, n)` contract scales seamlessly to the ray-triangle 3x3 case (the next drill), so practicing the shape arithmetic on the trivial 1x1 case is good muscle memory.

After solving, plugging `t` back into `O + t*D` exercises `ray-parametric-form` a second time. The `.unsqueeze(-1)` is the same broadcast trick from the standalone drill: `t: (NR,1)` vs `D: (NR,3)` lines up trailing dims so each ray's scalar parameter scales its own 3-vector direction.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx3',
        'subtopics': ["PyTorch: Batched linalg.solve", "Geometry: Ray parametric form"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()